<a href="https://colab.research.google.com/github/fukagai-takuya/gifu-ai/blob/main/gifu-ai-2026-09-06/RagPoc_Population_by_Pref_Age_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import requests
from pathlib import Path

# ==========================================
# e-Stat Excel ダウンロード
# ==========================================

URL = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000040479048&fileKind=0"

OUTPUT = Path(
    "/content/rag-poc/data/excel/Population_by_Pref_Age_2026.xlsx"
)

OUTPUT.parent.mkdir(parents=True, exist_ok=True)

response = requests.get(URL)
response.raise_for_status()

with open(OUTPUT, "wb") as f:
    f.write(response.content)

print(f"ダウンロード完了")
print(f"保存先: {OUTPUT}")
print(f"ファイルサイズ: {OUTPUT.stat().st_size:,} bytes")

ダウンロード完了
保存先: /content/rag-poc/data/excel/Population_by_Pref_Age_2026.xlsx
ファイルサイズ: 46,931 bytes


In [2]:
import openpyxl

path = "/content/rag-poc/data/excel/Population_by_Pref_Age_2026.xlsx"

wb = openpyxl.load_workbook(path, read_only=True, data_only=True)

print("Excel読み込み成功")
print("Sheet:", wb.sheetnames)

Excel読み込み成功
Sheet: ['年齢別人口（都道府県別）【総計】']


In [3]:
# ==========================================
# RAG PoC 環境構築
# ==========================================

!pip install -q \
    openpyxl \
    sentence-transformers \
    qdrant-client \
    transformers \
    accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 396.2/396.2 kB 16.5 MB/s eta 0:00:00


In [4]:
# ==========================================
# RAG PoC ディレクトリ作成
# ==========================================

from pathlib import Path

BASE_DIR = Path("/content/rag-poc")
DATA_DIR = BASE_DIR / "data" / "excel"
QDRANT_DIR = BASE_DIR / "qdrant_data"

DATA_DIR.mkdir(parents=True, exist_ok=True)
QDRANT_DIR.mkdir(parents=True, exist_ok=True)

print(f"BASE_DIR   : {BASE_DIR}")
print(f"DATA_DIR   : {DATA_DIR}")
print(f"QDRANT_DIR : {QDRANT_DIR}")

BASE_DIR   : /content/rag-poc
DATA_DIR   : /content/rag-poc/data/excel
QDRANT_DIR : /content/rag-poc/qdrant_data


In [5]:
# ==========================================
# Excel → JSON
# ==========================================

import json
import openpyxl

INPUT = "/content/rag-poc/data/excel/Population_by_Pref_Age_2026.xlsx"
OUTPUT = "/content/rag-poc/data/excel/Population_by_Pref_Age_2026.json"

wb = openpyxl.load_workbook(INPUT, data_only=True)
ws = wb.active

# 1行目：タイトル
title = ws.cell(1, 1).value

# 2行目：年齢階級
age_headers = [cell.value for cell in ws[2][3:]]

# 3行目：単位
units = [cell.value for cell in ws[3][3:]]

# 4～147行目：データ
records = []

for row in ws.iter_rows(min_row=4, max_row=147, values_only=True):
    record = {
        "団体コード": row[0],
        "都道府県名": row[1],
        "性別": row[2],
    }

    for age, value, unit in zip(age_headers, row[3:], units):
        record[age] = {
            "value": value,
            "unit": unit,
        }

    records.append(record)

# 148～150行目：注記
notes = [
    ws.cell(row, 1).value
    for row in range(148, 151)
    if ws.cell(row, 1).value
]

data = {
    "title": title,
    "source": INPUT,
    "notes": notes,
    "records": records,
}

with open(OUTPUT, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print(f"入力: {INPUT}")
print(f"出力: {OUTPUT}")
print(f"レコード数: {len(records)}")
print(f"注記数: {len(notes)}")

入力: /content/rag-poc/data/excel/Population_by_Pref_Age_2026.xlsx
出力: /content/rag-poc/data/excel/Population_by_Pref_Age_2026.json
レコード数: 144
注記数: 3


In [6]:
# ==========================================
# JSON → Chunk
# ==========================================

import json

INPUT = "/content/rag-poc/data/excel/Population_by_Pref_Age_2026.json"
OUTPUT = "/content/rag-poc/data/excel/Population_by_Pref_Age_2026-chunks.json"

with open(INPUT, "r", encoding="utf-8") as f:
    data = json.load(f)

chunks = []

for chunk_id, record in enumerate(data["records"]):
    lines = [
        data["title"],
        "",
        f"都道府県名：{record['都道府県名']}",
        f"性別：{record['性別']}",
        "",
    ]

    for key, item in record.items():
        if key in ["団体コード", "都道府県名", "性別"]:
            continue

        lines.append(
            f"{key}：{item['value']:,}{item['unit']}"
        )

    text = "\n".join(lines)

    chunks.append({
        "chunk_id": chunk_id,
        "text": text,
        "metadata": {
            "source": data["source"],
            "title": data["title"],
            "団体コード": record["団体コード"],
            "都道府県名": record["都道府県名"],
            "性別": record["性別"],
        }
    })

# 注記もChunkとして追加
for note in data["notes"]:
    chunk_id = len(chunks)

    text = f"{data['title']}\n\n注記：\n{note}"

    chunks.append({
        "chunk_id": chunk_id,
        "text": text,
        "metadata": {
            "source": data["source"],
            "title": data["title"],
            "type": "note"
        }
    })

with open(OUTPUT, "w", encoding="utf-8") as f:
    json.dump(chunks, f, ensure_ascii=False, indent=2)

print(f"入力: {INPUT}")
print(f"出力: {OUTPUT}")
print(f"Chunk数: {len(chunks)}")

入力: /content/rag-poc/data/excel/Population_by_Pref_Age_2026.json
出力: /content/rag-poc/data/excel/Population_by_Pref_Age_2026-chunks.json
Chunk数: 147


In [8]:
# ==========================================
# BGE-M3 Embedding
# ==========================================

import json
from sentence_transformers import SentenceTransformer

INPUT = "/content/rag-poc/data/excel/Population_by_Pref_Age_2026-chunks.json"
OUTPUT = "/content/rag-poc/data/excel/Population_by_Pref_Age_2026-embeddings.json"

MODEL_NAME = "BAAI/bge-m3"

# BGE-M3をGPUで読み込み
print("BGE-M3を読み込んでいます...")

embed_model = SentenceTransformer(
    MODEL_NAME,
    device="cuda",
)

# Chunk読み込み
with open(INPUT, "r", encoding="utf-8") as f:
    chunks = json.load(f)

texts = [chunk["text"] for chunk in chunks]

print(f"Chunk数: {len(texts)}")

# Embedding
embeddings = embed_model.encode(
    texts,
    batch_size=8,
    show_progress_bar=True,
    normalize_embeddings=True,
)

# 保存
results = []

for chunk, embedding in zip(chunks, embeddings):
    results.append({
        "chunk_id": chunk["chunk_id"],
        "text": chunk["text"],
        "metadata": chunk["metadata"],
        "embedding": embedding.tolist(),
    })

with open(OUTPUT, "w", encoding="utf-8") as f:
    json.dump(
        results,
        f,
        ensure_ascii=False,
        indent=2,
    )

print(f"入力: {INPUT}")
print(f"出力: {OUTPUT}")
print(f"Chunk数: {len(results)}")
print(f"Embedding次元数: {len(results[0]['embedding'])}")

BGE-M3を読み込んでいます...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Chunk数: 147


Batches:   0%|          | 0/19 [00:00<?, ?it/s]

入力: /content/rag-poc/data/excel/Population_by_Pref_Age_2026-chunks.json
出力: /content/rag-poc/data/excel/Population_by_Pref_Age_2026-embeddings.json
Chunk数: 147
Embedding次元数: 1024


In [9]:
# ==========================================
# BGE-M3解放
# ==========================================

import gc
import torch

del embed_model

gc.collect()
torch.cuda.empty_cache()

print("BGE-M3を解放しました。")
print(f"GPUメモリ使用量: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

BGE-M3を解放しました。
GPUメモリ使用量: 2.12 GB


In [10]:
# ==========================================
# GPUメモリ確認
# ==========================================

!nvidia-smi

Fri Aug 21 02:04:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   56C    P0             29W /   70W |    2331MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# ==========================================
# Qdrant Collection作成・登録
# ==========================================

import json
from pathlib import Path

from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

EMBEDDINGS_FILE = Path(
    "/content/rag-poc/data/excel/Population_by_Pref_Age_2026-embeddings.json"
)

COLLECTION_NAME = "population_age_2026"
VECTOR_SIZE = 1024

QDRANT_DIR = "/content/rag-poc/qdrant_data"

# Qdrant Local
client = QdrantClient(
    path=QDRANT_DIR
)

# Embeddings JSON読み込み
with open(EMBEDDINGS_FILE, "r", encoding="utf-8") as f:
    chunks = json.load(f)

print(f"読み込んだChunk数: {len(chunks)}")

# Collection作成
if client.collection_exists(COLLECTION_NAME):

    print(
        f"Collection '{COLLECTION_NAME}' は既に存在します。"
    )

else:

    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(
            size=VECTOR_SIZE,
            distance=Distance.COSINE,
        ),
    )

    print(
        f"Collection '{COLLECTION_NAME}' を作成しました。"
    )

# Qdrant Point作成
points = []

for chunk in chunks:

    points.append(
        PointStruct(
            id=chunk["chunk_id"],
            vector=chunk["embedding"],
            payload={
                "text": chunk["text"],
                "metadata": chunk["metadata"],
            },
        )
    )

# Qdrantへ登録
client.upsert(
    collection_name=COLLECTION_NAME,
    points=points,
)

print(
    f"{len(points)}個のChunkをQdrantへ登録しました。"
)

# 登録結果確認
collection_info = client.get_collection(
    collection_name=COLLECTION_NAME
)

print()
print(f"Collection: {COLLECTION_NAME}")
print(
    f"登録されたVector数: "
    f"{collection_info.points_count}"
)